# PPO: Proximal Policy Optimization

PPO is a policy-gradient method that keeps updates stable by clipping the probability ratio between the new and old policy. It avoids destructively large steps while still squeezing out most of the gradient signal.

## The loss

```
L_CLIP = E[ min(r_t * A_t,  clip(r_t, 1-ε, 1+ε) * A_t) ]
```

where `r_t = π_new(a|s) / π_old(a|s)` is the probability ratio and `A_t` is the advantage estimated via GAE.

The value and entropy terms are added to keep the critic honest and exploration alive:

```
L = -L_CLIP + 0.5 * L_value - 0.01 * H
```

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym

from src.agents.ppo import PPOAgent
from src.utils.seeding import set_seed

set_seed(42)
env = gym.make('CartPole-v1')
obs_dim = env.observation_space.shape[0]
n_actions = env.action_space.n

agent = PPOAgent(
    obs_dim=obs_dim, n_actions=n_actions,
    hidden_dims=(64, 64), lr=3e-4, gamma=0.99,
    gae_lambda=0.95, clip_eps=0.2,
    value_coef=0.5, entropy_coef=0.01,
    n_epochs=4, batch_size=32,
)
print('PPOAgent ready')

In [ ]:
rollout_rewards = []
policy_losses = []

for update in range(20):
    rollout = agent.collect_rollout(env, rollout_steps=64)
    loss = agent.update(rollout)
    total_r = sum(rollout['rewards'])
    rollout_rewards.append(total_r)
    policy_losses.append(loss)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(rollout_rewards)
ax1.set_title('Total rollout reward (PPO, 20 updates)')
ax1.set_xlabel('Update')
ax2.plot(policy_losses)
ax2.set_title('Total PPO loss per update')
ax2.set_xlabel('Update')
plt.tight_layout()
plt.show()

In [ ]:
# Visualise the GAE advantage signal from a single rollout
rollout = agent.collect_rollout(env, rollout_steps=128)
advantages, returns = agent.compute_gae(rollout['rewards'], rollout['values'], rollout['dones'])

plt.figure(figsize=(12, 3))
plt.plot(advantages, label='GAE advantage')
plt.plot(returns, label='Return', alpha=0.7)
plt.axhline(0, color='k', linewidth=0.5)
plt.xlabel('Timestep')
plt.title('Advantage and return estimates from a single 128-step rollout')
plt.legend()
plt.tight_layout()
plt.show()

Run the full PPO training with `python -m src.training.trainer --config-name ppo_cartpole` for a 100-update run.